# Step 3 — Nitrogen pollution cost accounting

This notebook estimates livestock-attributable nitrogen surplus and monetizes nitrogen pollution damages using low, mid, and high social cost of nitrogen (SCN) scenarios.

## Inputs and outputs

**Inputs**

- `../data/raw/FAOSTAT_Nitrogen_balance_2000-2022.csv`

**Outputs**

- `../data/outputs/nitrogen_cost_livestock_country_year.csv`
- `../data/outputs/nitrogen_cost_livestock_global_year.csv`

Place the notebook in `code/` and keep the raw and output files in parallel `data/raw` and `data/outputs` folders.

## Notebook workflow

This notebook is organized into short executable sections so that each transformation step is visible in GitHub and easy for reviewers to follow.

## Analysis step

In [ ]:
import pandas as pd
import numpy as np

## FILE PATH

In [ ]:
NITROGEN_PATH = "./FAOSTAT_Nitrogen_balance_2000-2022.csv"

OUT_COUNTRY = "./nitrogen_cost_livestock_country_year.csv"
OUT_GLOBAL  = "./nitrogen_cost_livestock_global_year.csv"

## PARAMETERS

In [ ]:
FEED_SHARE = 0.33   # livestock cropland share

SCN_VALUES = {
    "low": 5,     # USD per kg N
    "mid": 10,
    "high": 20
}

KEYS = ["Area", "Area Code (M49)", "Year"]

## LOAD DATA

In [ ]:
nitrogen = pd.read_csv(NITROGEN_PATH)

## 1) CALCULATE TOTAL NITROGEN SURPLUS

In [ ]:
INPUT_ITEMS = {
    "Mineral fertilizers",
    "Manure applied",
    "Atmospheric deposition",
    "Biological fixation",
    "Seed",
}

OUTPUT_ITEMS = {
    "Crop harvest removal",
    "Crop residue removal",
}

n = nitrogen[nitrogen["Item"].isin(INPUT_ITEMS | OUTPUT_ITEMS)].copy()

## Ensure numeric

In [ ]:
n["Value"] = pd.to_numeric(n["Value"], errors="coerce")
n = n.dropna(subset=["Value"])

# Assign sign
n["signed_value_tonnesN"] = n["Value"]
n.loc[n["Item"].isin(OUTPUT_ITEMS), "signed_value_tonnesN"] *= -1.0

# Aggregate country-year surplus
n_surplus = (
    n.groupby(KEYS, as_index=False)["signed_value_tonnesN"]
     .sum()
     .rename(columns={"signed_value_tonnesN": "N_surplus_total_tonnesN"})
)

## 2) ALLOCATE 33% TO LIVESTOCK

In [ ]:
n_surplus["N_livestock_tonnesN"] = (
    n_surplus["N_surplus_total_tonnesN"] * FEED_SHARE
)

# Convert tonnes → kg (for monetization)
n_surplus["N_livestock_kg"] = (
    n_surplus["N_livestock_tonnesN"] * 1000.0
)

## 3) CREATE SCENARIO TABLE

In [ ]:
params = pd.DataFrame([
    {"scenario": s, "SCN_usd_per_kgN": SCN_VALUES[s]}
    for s in ["low", "mid", "high"]
])

## Cross join

In [ ]:
n_surplus["key"] = 1
params["key"] = 1

df = n_surplus.merge(params, on="key").drop(columns="key")

## 4) MONETIZE NITROGEN

In [ ]:
df["nitrogen_cost_usd"] = (
    df["N_livestock_kg"] *
    df["SCN_usd_per_kgN"]
)

## 5) SAVE COUNTRY-YEAR FILE

In [ ]:
df.to_csv(OUT_COUNTRY, index=False)

print("Saved country-year file:", OUT_COUNTRY)
print(df.head())

## 6) CREATE GLOBAL-YEAR FILE

In [ ]:
global_summary = (
    df.groupby(["scenario", "Year"], as_index=False)
      .agg(
          N_surplus_total_tonnesN=("N_surplus_total_tonnesN", "sum"),
          N_livestock_tonnesN=("N_livestock_tonnesN", "sum"),
          nitrogen_cost_usd=("nitrogen_cost_usd", "sum")
      )
)

global_summary.to_csv(OUT_GLOBAL, index=False)

print("Saved global-year file:", OUT_GLOBAL)
print(global_summary.head())